## Chroma CRUD OPERATIONS

Walks through create, read, update and delete with a local Chroma Vector Store

In [11]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings


# Ollama embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


In [6]:
collection_name="demo"
project_root = Path.cwd()
persists_directory = project_root / "chroma_langchain_db"

In [7]:
# starts fresh so CRUD flow produces the same result each time
if persists_directory.exists():
    shutil.rmtree(persists_directory)
    print("removed the old chroma directory")
else:
    print("No previous chroma directory was found")


No previous chroma directory was found


In [10]:
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persists_directory)
)
print("vector store is ready")

vector store is ready


In [13]:
# small helper functions
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)

    for index, doc in enumerate(docs, start=1):
        print(
            f"{index}: id={doc.id}"
            f"    topic={doc.metadata.get('topic')}"
            f"    doc_number={doc.metadata.get('doc_number')}"
        )

        print(
            f"    content={doc.page_content}"
        )

    print()

In [14]:
# create and insert examle documents
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [15]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [16]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1: id=0e756956-1edf-4fd7-a341-1102e1837704    topic=AI    doc_number=1
    content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2: id=7a3abc5c-1ee9-4421-a55e-ceab293aa868    topic=AI    doc_number=2
    content=AI systems can analyze patterns in data to support predictions and automation.
3: id=b9195900-9a03-44ff-ac76-e565db4724c6    topic=AI    doc_number=3
    content=Responsible AI development includes fairness, transparency, and safety checks.
4: id=5cea8c46-195c-4a39-a377-d6b2c7204ec9    topic=RAG    doc_number=4
    content=RAG combines retrieval with generation so the model can answer using external knowledge.
5: id=bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b    topic=RAG    doc_number=5
    content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6: id=07d51640-83dd-4b5d-ae21-f790d7944e39    topic=RAG    doc_number=6
    content=Vector stores are important in RAG beca

In [18]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:",document_ids)
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids: ['0e756956-1edf-4fd7-a341-1102e1837704', '7a3abc5c-1ee9-4421-a55e-ceab293aa868', 'b9195900-9a03-44ff-ac76-e565db4724c6', '5cea8c46-195c-4a39-a377-d6b2c7204ec9', 'bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b', '07d51640-83dd-4b5d-ae21-f790d7944e39', 'c44e6d81-1986-4010-98fc-66c3b4b69ee4', 'e57298a2-cf9e-486d-8aa9-d29b9346f9f0', 'b6f28847-43de-45b0-b2ae-3aab7518bb66', '726030ad-4c4e-4d33-a9df-b1f35bf80c63']
0e756956-1edf-4fd7-a341-1102e1837704
7a3abc5c-1ee9-4421-a55e-ceab293aa868
b9195900-9a03-44ff-ac76-e565db4724c6
5cea8c46-195c-4a39-a377-d6b2c7204ec9
bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b
07d51640-83dd-4b5d-ae21-f790d7944e39
c44e6d81-1986-4010-98fc-66c3b4b69ee4
e57298a2-cf9e-486d-8aa9-d29b9346f9f0
b6f28847-43de-45b0-b2ae-3aab7518bb66
726030ad-4c4e-4d33-a9df-b1f35bf80c63

Total inserted documents: 10


## Read the Stored Data Back

In [ ]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

{'ids': ['0e756956-1edf-4fd7-a341-1102e1837704',
  '7a3abc5c-1ee9-4421-a55e-ceab293aa868',
  'b9195900-9a03-44ff-ac76-e565db4724c6',
  '5cea8c46-195c-4a39-a377-d6b2c7204ec9',
  'bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b',
  '07d51640-83dd-4b5d-ae21-f790d7944e39',
  'c44e6d81-1986-4010-98fc-66c3b4b69ee4',
  'e57298a2-cf9e-486d-8aa9-d29b9346f9f0',
  'b6f28847-43de-45b0-b2ae-3aab7518bb66',
  '726030ad-4c4e-4d33-a9df-b1f35bf80c63'],
 'embeddings': array([[ 0.01127994,  0.09853448, -0.12941295, ..., -0.00943835,
         -0.06669365,  0.00182315],
        [-0.00344792,  0.05621187, -0.1512849 , ..., -0.05118925,
         -0.04473256,  0.02945543],
        [ 0.0730267 ,  0.08137438, -0.14276837, ..., -0.10128256,
         -0.03788133, -0.00550622],
        ...,
        [ 0.02318365,  0.02986862, -0.1187271 , ..., -0.03614221,
         -0.03005022, -0.03526313],
        [ 0.04195919,  0.08839762, -0.19457781, ..., -0.03785993,
         -0.0655752 , -0.01618887],
        [ 0.01923946,  0.07774293, 

In [22]:
print(raw_records["embeddings"].shape)
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

(10, 768)
Total records in collection: 10
First three ids from get():
0e756956-1edf-4fd7-a341-1102e1837704
7a3abc5c-1ee9-4421-a55e-ceab293aa868
b9195900-9a03-44ff-ac76-e565db4724c6


In [23]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1: id=e57298a2-cf9e-486d-8aa9-d29b9346f9f0    topic=LLM    doc_number=8
    content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2: id=b6f28847-43de-45b0-b2ae-3aab7518bb66    topic=Cricket    doc_number=9
    content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3: id=726030ad-4c4e-4d33-a9df-b1f35bf80c63    topic=Cricket    doc_number=10
    content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



## Run a Similarity Search

In [24]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [25]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1: id=5cea8c46-195c-4a39-a377-d6b2c7204ec9    topic=RAG    doc_number=4
    content=RAG combines retrieval with generation so the model can answer using external knowledge.
2: id=e57298a2-cf9e-486d-8aa9-d29b9346f9f0    topic=LLM    doc_number=8
    content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3: id=bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b    topic=RAG    doc_number=5
    content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [26]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='5cea8c46-195c-4a39-a377-d6b2c7204ec9', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.5347505807876587),
 (Document(id='e57298a2-cf9e-486d-8aa9-d29b9346f9f0', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.5732569694519043),
 (Document(id='bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  0.7347312569618225),
 (Document(id='07d51640-83dd-4b5d-ae21-f790d7944e39', metadata={'topic': 'RAG', 'doc_number': 6}, page_content='Vector stores are important in RAG because they make semantic search over embedded documents possible.'),
  0.7820466160774231)]

## Update Existing Documents

In [27]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['5cea8c46-195c-4a39-a377-d6b2c7204ec9',
 'e57298a2-cf9e-486d-8aa9-d29b9346f9f0']

In [28]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1: id=5cea8c46-195c-4a39-a377-d6b2c7204ec9    topic=RAG    doc_number=4
    content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2: id=e57298a2-cf9e-486d-8aa9-d29b9346f9f0    topic=LLM    doc_number=8
    content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [29]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

In [30]:
updated_raw_records = vector_store.get(ids=ids_to_update)

In [31]:
print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=5cea8c46-195c-4a39-a377-d6b2c7204ec9
metadata={'topic': 'RAG', 'doc_number': 4}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=e57298a2-cf9e-486d-8aa9-d29b9346f9f0
metadata={'topic': 'LLM', 'doc_number': 8}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [32]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [33]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1: id=5cea8c46-195c-4a39-a377-d6b2c7204ec9    topic=RAG    doc_number=4
    content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2: id=bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b    topic=RAG    doc_number=5
    content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## Delete Documents

In [34]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['b6f28847-43de-45b0-b2ae-3aab7518bb66',
 '726030ad-4c4e-4d33-a9df-b1f35bf80c63']

In [35]:
vector_store.delete(ids=ids_to_delete)

In [37]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
